# 3. Machine Learning for Classification
We'll use logistic regression to predict churn.

In [7]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

## 3.1 Churn prediction project
[Dataset](https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/refs/heads/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv)

In [8]:
dataset_path = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/refs/heads/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

In [9]:
!wget $dataset_path -O data-week-3.csv

--2024-10-16 08:42:33--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/refs/heads/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Résolution de raw.githubusercontent.com (raw.githubusercontent.com)… 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connexion à raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443… connecté.
requête HTTP transmise, en attente de la réponse… 200 OK
Taille : 977501 (955K) [text/plain]
Sauvegarde en : « data-week-3.csv »

data-week-3.csv     100%[===================>] 954.59K   572KB/s    ds 1.7s    

2024-10-16 08:42:35 (572 KB/s) — « data-week-3.csv » sauvegardé [977501/977501]



## 3.2. Data preparation
- Download the data, read it with pandas
- Look at the data
- Make column names and values look uniform
- Check if all the columns read correctly
- Check if the churn variable needs any preparation

In [13]:
df = pd.read_csv('data-week-3.csv')

In [14]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [15]:
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

In [21]:
df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


Consistant values and column names

In [19]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for col in categorical_columns:
    df[col] = df[col].str.lower().str.replace(' ', '_')

In [20]:
df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


Correct type: Float

In [22]:
df['totalcharges']

0         29.85
1        1889.5
2        108.15
3       1840.75
4        151.65
         ...   
7038     1990.5
7039     7362.9
7040     346.45
7041      306.6
7042     6844.5
Name: totalcharges, Length: 7043, dtype: object

In [23]:
pd.to_numeric(df['totalcharges'])

ValueError: Unable to parse string "_" at position 488

In [25]:
df['totalcharges'][488]

'_'

In [27]:
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')

In [ ]:
df.totalcharges = df.totalcharges.fillna(0)

Correct type: Boolean as Integer

In [29]:
df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


In [42]:
columns = df.columns
for column in columns:
    print(f'{column}: {len(df[column].value_counts())}')

customerid: 7043
gender: 2
seniorcitizen: 2
partner: 2
dependents: 2
tenure: 73
phoneservice: 2
multiplelines: 3
internetservice: 3
onlinesecurity: 3
onlinebackup: 3
deviceprotection: 3
techsupport: 3
streamingtv: 3
streamingmovies: 3
contract: 3
paperlessbilling: 2
paymentmethod: 4
monthlycharges: 1585
totalcharges: 6530
churn: 2


The other 'yes/no' columns will be later one-hot encoded, so we only need to worry about our y column: churn

In [43]:
df.churn.value_counts()

churn
no     5174
yes    1869
Name: count, dtype: int64

In [45]:
(df.churn == 'yes').astype('int')

0       0
1       0
2       1
3       0
4       1
       ..
7038    0
7039    0
7040    0
7041    1
7042    0
Name: churn, Length: 7043, dtype: int64

In [46]:
df.churn = (df.churn == 'yes').astype('int')

## 3.3. Setting up the validation framework
- Perform the train/validation/test split with Scikit-Learn

In [47]:
from sklearn.model_selection import train_test_split

In [48]:
train_test_split?

Signature:
train_test_split(
    *arrays,
    test_size=None,
    train_size=None,
    random_state=None,
    shuffle=True,
    stratify=None,
)
Docstring:
Split arrays or matrices into random train and test subsets.

Quick utility that wraps input validation,
``next(ShuffleSplit().split(X, y))``, and application to input data
into a single call for splitting (and optionally subsampling) data into a
one-liner.

Read more in the :ref:`User Guide <cross_validation>`.

Parameters
----------
*arrays : sequence of indexables with same length / shape[0]
    Allowed inputs are lists, numpy arrays, scipy-sparse
    matrices or pandas dataframes.

test_size : float or int, default=None
    If float, should be between 0.0 and 1.0 and represent the proportion
    of the dataset to include in the test split. If int, represents the
    absolute number of test samples. If None, the value is set to the
    complement of the train size. If ``train_size`` is also None, it will
    be set to 0.25.

trai

In [57]:
[df_full_train, df_test] = train_test_split(df, test_size=0.2, random_state=1)
[df_train, df_val] = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [58]:
len(df_train), len(df_test), len(df_val)

(4225, 1409, 1409)

In [59]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [60]:
y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

In [62]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

In [64]:
def train_test_val_split(df, train_size, test_size, y_column, random_state):
    [df_full_train, df_test] = train_test_split(df, test_size=test_size, random_state=1)
    val_size = (1 - train_size - test_size) / (1 - test_size)
    [df_train, df_val] = train_test_split(df_full_train, test_size=val_size, random_state=1)
    
    df_full_train = df_full_train.reset_index(drop=True)
    df_train = df_train.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)
    
    y_full_train = df_full_train[y_column].values
    y_train = df_train[y_column].values
    y_val = df_val[y_column].values
    y_test = df_test[y_column].values
    
    del df_train['churn']
    del df_val['churn']
    del df_test['churn']
    
    return df_full_train, y_full_train, df_train, y_train, df_test, y_test, df_val, y_val

## 3.4. EDA
- Check missing values
- Look at the target variable (`churn`)
- Look at numerical and categorical variables

In [69]:
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        8
churn               0
dtype: int64

In [71]:
df_full_train.churn.value_counts(normalize=True)

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [72]:
global_churn_rate = df_full_train.churn.mean()
round(global_churn_rate, 2)

0.27

## 3.5. Feature importance: Churn rate and risk ratio
Feature importance analysis (part of EDA) - identifying which features affect our target variable
- Churn rate
- Risk ratio
- Mutual information

### Churn rate

### Risk ratio

## 3.6. Feature importance: Mutual information
Mutual information - concept from information theory, it tells us how much we can learn about one variable if we know the value of another.

[Mutual Information - Wikipedia](https://en.wikipedia.org/wiki/Mutual_information)

## 3.7. Feature importance: Correlation
How about numerical columns? Correlation coefficient.

## 3.8. One hot encoding
Use Scikit-Learn to encode categorical features

## 3.9. Logistic regression
- Binary classification
- Linear vs logistic regression

## 3.10. Training logistic regression with Scikit-Learn
- Train a model with Scikit-Learn
- Apply it to the validation dataset
- Calculate the accuracy

## 3.11. Model interpretation
- Look at the coefficients
- Train a smaller model with fewer features

## 3.12. Using the model

## 3.13. Summary
- Feature importance - risks, mutual information, correlation
- One-hot encoding can be implemented with `DictVectorizer`
- Logistic regression - linear model like linear regression
- Output of `log` regression - probability
- Interpretation of weights is similar to linear regression